In [2]:
import os
from pathlib import Path
import re

In [3]:
def check_hypothesis_formation(content):
    """Check if hypothesis formation file is complete."""
    return content.strip().endswith('</pattern_summary>')

def check_hypothesis_validation(content):
    """Check if hypothesis validation file is complete."""
    return '</validated_pattern>' in content

def extract_sample_scores(content):
    """Extract sample scores from final code file."""
    scores = {}
    
    # Look for the sample scores section
    # Pattern: "Sample 0: 1.00 ✓" or "Sample 0: 1.00" or "Sample 0: 0.67"
    pattern = r'Sample (\d+):\s+([\d.]+)'
    matches = re.findall(pattern, content)
    
    for sample_num, score in matches:
        scores[int(sample_num)] = float(score)
    
    return scores

def check_final_code(content):
    """Check if final code file is complete and extract sample scores."""
    # Look for the test performance section that indicates completion
    markers = [
        'Test Set Performance:',
        'Best candidate (sample',
        'SELECTED: Sample'
    ]
    is_complete = all(marker in content for marker in markers)
    
    sample_scores = extract_sample_scores(content)
    
    return is_complete, sample_scores


In [5]:
def analyze_task_completion(task_dir):
    """Analyze completion status for a single task."""
    task_id = task_dir.name
    results = {
        'task_id': task_id,
        'hypothesis_formation': {},  # sample_num -> complete_status
        'hypothesis_validation': {},  # sample_num -> complete_status
        'final_code': None,
        'final_code_scores': {},
        'repair_code': None,
        'repair_code_scores': {}
    }
        # Check all files in the task directory
    for file_path in task_dir.iterdir():
        if not file_path.is_file():
            continue
            
        try:
            content = file_path.read_text(encoding='utf-8')
            filename = file_path.name
            
            # Parse filename to extract sample number and phase
            # Format: {task_id}_sample{N}_phase2a_hypothesis.txt
            #         {task_id}_sample{N}_phase2b_validation.txt
            #         {task_id}_selection_summary.txt
            #         {task_id}_repair_selection_summary.txt
            
            if '_sample' in filename and '_phase2a_' in filename:
                # Hypothesis formation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    results['hypothesis_formation'][sample_num] = check_hypothesis_formation(content)
            
            elif '_sample' in filename and '_phase2b_' in filename:
                # Hypothesis validation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    results['hypothesis_validation'][sample_num] = check_hypothesis_validation(content)
            
            elif '_selection_summary.txt' in filename:
                if '_repair_selection_summary.txt' in filename:
                    # Repair selection summary
                    is_complete, scores = check_final_code(content)
                    results['repair_code'] = is_complete
                    results['repair_code_scores'] = scores
                else:
                    # Regular selection summary
                    is_complete, scores = check_final_code(content)
                    results['final_code'] = is_complete
                    results['final_code_scores'] = scores
        
        except Exception as e:
            print(f"  Error reading {file_path.name}: {e}")
    
    return results

In [6]:
def format_sample_dict(sample_dict, expected_samples=4):
    """Format sample completion status."""
    if not sample_dict:
        return "❌ Not found"
    
    status_parts = []
    all_complete = True
    
    for i in range(expected_samples):
        if i in sample_dict:
            if sample_dict[i]:
                status_parts.append(f"S{i}:✓")
            else:
                status_parts.append(f"S{i}:✗")
                all_complete = False
        else:
            status_parts.append(f"S{i}:❌")
            all_complete = False
    
    status_str = " ".join(status_parts)
    summary = "✓ All complete" if all_complete else "✗ Incomplete"
    return f"{status_str} ({summary})"


In [7]:
def format_scores(scores, expected_samples=4):
    """Format sample scores."""
    if not scores:
        return "No scores found"
    
    score_parts = []
    for i in range(expected_samples):
        if i in scores:
            score_parts.append(f"S{i}:{scores[i]:.2f}")
        else:
            score_parts.append(f"S{i}:N/A")
    
    return " ".join(score_parts)

def format_status(status):
    """Format the status for display."""
    if status is None:
        return "❌ Not found"
    elif status is True:
        return "✓ Complete"
    else:
        return "✗ Incomplete"


In [9]:
def get_project_root():
    """Find the project root directory."""
    # Try to get the directory of the current script
    try:
        # If __file__ is available
        current_file = Path(__file__).resolve()
        current_dir = current_file.parent
    except NameError:
        # If __file__ is not available, use current working directory
        current_dir = Path.cwd()
    
    # If we're in src/, go up one level
    if current_dir.name == 'src':
        return current_dir.parent
    
    # Otherwise, assume we're already at project root
    return current_dir


In [10]:
def main():
    # Navigate from src/ to logs directory
    project_root = get_project_root()
    logs_dir = project_root / 'logs' / 'grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728'
    
    if not logs_dir.exists():
        print(f"Error: Directory not found: {logs_dir}")
        print(f"Project root detected as: {project_root}")
        print(f"Current working directory: {Path.cwd()}")
        return
    
    print(f"Analyzing tasks in: {logs_dir}\n")
    print("=" * 100)
    
    # Collect all task directories
    task_dirs = [d for d in logs_dir.iterdir() if d.is_dir()]
    task_dirs.sort()
    
    incomplete_tasks = []
    
    for task_dir in task_dirs:
        results = analyze_task_completion(task_dir)
        task_id = results['task_id']
        
        # Check if all components are complete
        hyp_form_complete = len(results['hypothesis_formation']) == 4 and all(results['hypothesis_formation'].values())
        hyp_val_complete = len(results['hypothesis_validation']) == 4 and all(results['hypothesis_validation'].values())
        final_code_complete = results['final_code'] is True
        
        all_complete = hyp_form_complete and hyp_val_complete and final_code_complete
        
        if not all_complete:
            incomplete_tasks.append(results)
        
        # Print status
        status_symbol = "✓" if all_complete else "✗"
        print(f"{status_symbol} Task {task_id}:")
        print(f"  Hypothesis Formation (Phase 2A): {format_sample_dict(results['hypothesis_formation'])}")
        print(f"  Hypothesis Validation (Phase 2B): {format_sample_dict(results['hypothesis_validation'])}")
        print(f"  Final Code: {format_status(results['final_code'])}")
        if results['final_code_scores']:
            print(f"    Sample Scores: {format_scores(results['final_code_scores'])}")
        
        # Print repair code info if exists
        if results['repair_code'] is not None:
            print(f"  Repair Code: {format_status(results['repair_code'])}")
            if results['repair_code_scores']:
                print(f"    Sample Scores: {format_scores(results['repair_code_scores'])}")
        
        print()
    
    # Summary
    print("=" * 100)
    print(f"\nSummary:")
    print(f"  Total tasks: {len(task_dirs)}")
    print(f"  Complete: {len(task_dirs) - len(incomplete_tasks)}")
    print(f"  Incomplete: {len(incomplete_tasks)}")
    
    if incomplete_tasks:
        print("\nIncomplete tasks details:")
        for result in incomplete_tasks:
            missing = []
            
            # Check what's missing for hypothesis formation
            hyp_form = result['hypothesis_formation']
            if len(hyp_form) < 4:
                missing_samples = [i for i in range(4) if i not in hyp_form]
                missing.append(f"2a(missing S{missing_samples})")
            elif not all(hyp_form.values()):
                incomplete_samples = [i for i in range(4) if i in hyp_form and not hyp_form[i]]
                missing.append(f"2a(incomplete S{incomplete_samples})")
            
            # Check what's missing for hypothesis validation
            hyp_val = result['hypothesis_validation']
            if len(hyp_val) < 4:
                missing_samples = [i for i in range(4) if i not in hyp_val]
                missing.append(f"2b(missing S{missing_samples})")
            elif not all(hyp_val.values()):
                incomplete_samples = [i for i in range(4) if i in hyp_val and not hyp_val[i]]
                missing.append(f"2b(incomplete S{incomplete_samples})")
            
            # Check final code
            if not result['final_code']:
                missing.append('code')
            
            print(f"  {result['task_id']}: {', '.join(missing) if missing else 'unknown issue'}")

if __name__ == '__main__':
    main()

Analyzing tasks in: /home/te0245/llms_ftw/logs/grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728


Summary:
  Total tasks: 0
  Complete: 0
  Incomplete: 0


In [11]:
import os
from pathlib import Path
import re
import sys

def check_hypothesis_formation(content):
    """Check if hypothesis formation file is complete."""
    return content.strip().endswith('</pattern_summary>')

def check_hypothesis_validation(content):
    """Check if hypothesis validation file is complete."""
    return '</validated_pattern>' in content

def extract_sample_scores(content):
    """Extract sample scores from final code file."""
    scores = {}
    
    # Look for the sample scores section
    # Pattern: "Sample 0: 1.00 ✓" or "Sample 0: 1.00" or "Sample 0: 0.67"
    pattern = r'Sample (\d+):\s+([\d.]+)'
    matches = re.findall(pattern, content)
    
    for sample_num, score in matches:
        scores[int(sample_num)] = float(score)
    
    return scores

def check_final_code(content):
    """Check if final code file is complete and extract sample scores."""
    # Look for the test performance section that indicates completion
    markers = [
        'Test Set Performance:',
        'Best candidate (sample',
        'SELECTED: Sample'
    ]
    is_complete = all(marker in content for marker in markers)
    
    sample_scores = extract_sample_scores(content)
    
    return is_complete, sample_scores

def analyze_task_completion(task_dir):
    """Analyze completion status for a single task."""
    task_id = task_dir.name
    results = {
        'task_id': task_id,
        'hypothesis_formation': {},  # sample_num -> complete_status
        'hypothesis_validation': {},  # sample_num -> complete_status
        'final_code': None,
        'final_code_scores': {},
        'repair_code': None,
        'repair_code_scores': {}
    }
    
    # Check all files in the task directory
    for file_path in task_dir.iterdir():
        if not file_path.is_file():
            continue
            
        try:
            content = file_path.read_text(encoding='utf-8')
            filename = file_path.name
            
            # Parse filename to extract sample number and phase
            # Format: {task_id}_sample{N}_phase2a_hypothesis.txt
            #         {task_id}_sample{N}_phase2b_validation.txt
            #         {task_id}_selection_summary.txt
            #         {task_id}_repair_selection_summary.txt
            
            if '_sample' in filename and '_phase2a_' in filename:
                # Hypothesis formation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    results['hypothesis_formation'][sample_num] = check_hypothesis_formation(content)
            
            elif '_sample' in filename and '_phase2b_' in filename:
                # Hypothesis validation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    results['hypothesis_validation'][sample_num] = check_hypothesis_validation(content)
            
            elif '_selection_summary.txt' in filename:
                if '_repair_selection_summary.txt' in filename:
                    # Repair selection summary
                    is_complete, scores = check_final_code(content)
                    results['repair_code'] = is_complete
                    results['repair_code_scores'] = scores
                else:
                    # Regular selection summary
                    is_complete, scores = check_final_code(content)
                    results['final_code'] = is_complete
                    results['final_code_scores'] = scores
        
        except Exception as e:
            print(f"  Error reading {file_path.name}: {e}")
    
    return results

def format_sample_dict(sample_dict, expected_samples=4):
    """Format sample completion status."""
    if not sample_dict:
        return "❌ Not found"
    
    status_parts = []
    all_complete = True
    
    for i in range(expected_samples):
        if i in sample_dict:
            if sample_dict[i]:
                status_parts.append(f"S{i}:✓")
            else:
                status_parts.append(f"S{i}:✗")
                all_complete = False
        else:
            status_parts.append(f"S{i}:❌")
            all_complete = False
    
    status_str = " ".join(status_parts)
    summary = "✓ All complete" if all_complete else "✗ Incomplete"
    return f"{status_str} ({summary})"

def format_scores(scores, expected_samples=4):
    """Format sample scores."""
    if not scores:
        return "No scores found"
    
    score_parts = []
    for i in range(expected_samples):
        if i in scores:
            score_parts.append(f"S{i}:{scores[i]:.2f}")
        else:
            score_parts.append(f"S{i}:N/A")
    
    return " ".join(score_parts)

def format_status(status):
    """Format the status for display."""
    if status is None:
        return "❌ Not found"
    elif status is True:
        return "✓ Complete"
    else:
        return "✗ Incomplete"

def get_project_root():
    """Find the project root directory."""
    # Try to get the directory of the current script
    try:
        # If __file__ is available
        current_file = Path(__file__).resolve()
        current_dir = current_file.parent
    except NameError:
        # If __file__ is not available, use current working directory
        current_dir = Path.cwd()
    
    # If we're in src/, go up one level
    if current_dir.name == 'src':
        return current_dir.parent
    
    # Otherwise, assume we're already at project root
    return current_dir

def main():
    # Navigate to logs directory
    project_root = get_project_root()
    logs_dir = project_root / 'logs' / 'grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728'
    
    print(f"DEBUG: Project root: {project_root}")
    print(f"DEBUG: Current working directory: {Path.cwd()}")
    print(f"DEBUG: Looking for logs at: {logs_dir}")
    print(f"DEBUG: Logs dir exists: {logs_dir.exists()}")
    print()
    
    if not logs_dir.exists():
        print(f"Error: Directory not found: {logs_dir}")
        print(f"\nLet's explore what's available:")
        
        # Check if logs directory exists at all
        logs_base = project_root / 'logs'
        if logs_base.exists():
            print(f"\nLogs directory exists at: {logs_base}")
            print("Available subdirectories:")
            for item in logs_base.iterdir():
                if item.is_dir():
                    print(f"  - {item.name}")
        else:
            print(f"Logs directory doesn't exist at: {logs_base}")
        
        return
    
    print(f"Analyzing tasks in: {logs_dir}\n")
    
    # List what's inside the logs directory
    print("DEBUG: Contents of logs directory:")
    all_items = list(logs_dir.iterdir())
    for item in all_items:
        print(f"  {'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")
    print()
    
    print("=" * 100)
    
    # Collect all task directories
    task_dirs = [d for d in logs_dir.iterdir() if d.is_dir()]
    task_dirs.sort()
    
    print(f"DEBUG: Found {len(task_dirs)} task directories")
    if task_dirs:
        print("DEBUG: First few task dirs:", [d.name for d in task_dirs[:5]])
    print()
    
    if not task_dirs:
        print("No task directories found!")
        print("\nMaybe the files are directly in this directory?")
        
        # Check if there are txt files directly
        txt_files = [f for f in logs_dir.iterdir() if f.is_file() and f.suffix == '.txt']
        if txt_files:
            print(f"Found {len(txt_files)} .txt files directly in logs directory")
            print("First few files:", [f.name for f in txt_files[:5]])
        
        return
    
    incomplete_tasks = []
    
    for task_dir in task_dirs:
        results = analyze_task_completion(task_dir)
        task_id = results['task_id']
        
        # Check if all components are complete
        hyp_form_complete = len(results['hypothesis_formation']) == 4 and all(results['hypothesis_formation'].values())
        hyp_val_complete = len(results['hypothesis_validation']) == 4 and all(results['hypothesis_validation'].values())
        final_code_complete = results['final_code'] is True
        
        all_complete = hyp_form_complete and hyp_val_complete and final_code_complete
        
        if not all_complete:
            incomplete_tasks.append(results)
        
        # Print status
        status_symbol = "✓" if all_complete else "✗"
        print(f"{status_symbol} Task {task_id}:")
        print(f"  Hypothesis Formation (Phase 2A): {format_sample_dict(results['hypothesis_formation'])}")
        print(f"  Hypothesis Validation (Phase 2B): {format_sample_dict(results['hypothesis_validation'])}")
        print(f"  Final Code: {format_status(results['final_code'])}")
        if results['final_code_scores']:
            print(f"    Sample Scores: {format_scores(results['final_code_scores'])}")
        
        # Print repair code info if exists
        if results['repair_code'] is not None:
            print(f"  Repair Code: {format_status(results['repair_code'])}")
            if results['repair_code_scores']:
                print(f"    Sample Scores: {format_scores(results['repair_code_scores'])}")
        
        print()
    
    # Summary
    print("=" * 100)
    print(f"\nSummary:")
    print(f"  Total tasks: {len(task_dirs)}")
    print(f"  Complete: {len(task_dirs) - len(incomplete_tasks)}")
    print(f"  Incomplete: {len(incomplete_tasks)}")
    
    if incomplete_tasks:
        print("\nIncomplete tasks details:")
        for result in incomplete_tasks:
            missing = []
            
            # Check what's missing for hypothesis formation
            hyp_form = result['hypothesis_formation']
            if len(hyp_form) < 4:
                missing_samples = [i for i in range(4) if i not in hyp_form]
                missing.append(f"2a(missing S{missing_samples})")
            elif not all(hyp_form.values()):
                incomplete_samples = [i for i in range(4) if i in hyp_form and not hyp_form[i]]
                missing.append(f"2a(incomplete S{incomplete_samples})")
            
            # Check what's missing for hypothesis validation
            hyp_val = result['hypothesis_validation']
            if len(hyp_val) < 4:
                missing_samples = [i for i in range(4) if i not in hyp_val]
                missing.append(f"2b(missing S{missing_samples})")
            elif not all(hyp_val.values()):
                incomplete_samples = [i for i in range(4) if i in hyp_val and not hyp_val[i]]
                missing.append(f"2b(incomplete S{incomplete_samples})")
            
            # Check final code
            if not result['final_code']:
                missing.append('code')
            
            print(f"  {result['task_id']}: {', '.join(missing) if missing else 'unknown issue'}")

if __name__ == '__main__':
    main()

DEBUG: Project root: /home/te0245/llms_ftw
DEBUG: Current working directory: /home/te0245/llms_ftw/src
DEBUG: Looking for logs at: /home/te0245/llms_ftw/logs/grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728
DEBUG: Logs dir exists: True

Analyzing tasks in: /home/te0245/llms_ftw/logs/grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728

DEBUG: Contents of logs directory:
  [FILE] 5545f144_sample2_phase2a_hypothesis.txt
  [FILE] 271d71e2_sample1_phase2a_hypothesis.txt
  [FILE] 332f06d7_selection_summary.txt
  [FILE] b6f77b65_sample0_phase2a_hypothesis.txt
  [FILE] 7b0280bc_sample3_phase2a_hypothesis.txt
  [FILE] b10624e5_selection_summary.txt
  [FILE] 7b80bb43_selection_summary.txt
  [FILE] f560132c_sample0_phase2b_validation.txt
  [FILE] 8698868d_sample1_phase2a_hypothesis.txt
  [FILE] 4c3d4a41_sample0_phase2a_hypothesis.txt
  [FILE] a47bf94d_sample3_phase2a_hypothesis.txt
  [FILE] 7b3084d4_repair_selection_summary.txt
  [FILE] 89565ca0_sample2_phase2b_validation.txt
  [FILE] 5545f1

In [12]:
import os
from pathlib import Path
import re
from collections import defaultdict

def check_hypothesis_formation(content):
    """Check if hypothesis formation file is complete."""
    return content.strip().endswith('</pattern_summary>')

def check_hypothesis_validation(content):
    """Check if hypothesis validation file is complete."""
    return '</validated_pattern>' in content

def extract_sample_scores(content):
    """Extract sample scores from final code file."""
    scores = {}
    
    # Look for the sample scores section
    # Pattern: "Sample 0: 1.00 ✓" or "Sample 0: 1.00" or "Sample 0: 0.67"
    pattern = r'Sample (\d+):\s+([\d.]+)'
    matches = re.findall(pattern, content)
    
    for sample_num, score in matches:
        scores[int(sample_num)] = float(score)
    
    return scores

def check_final_code(content):
    """Check if final code file is complete and extract sample scores."""
    # Look for the test performance section that indicates completion
    markers = [
        'Test Set Performance:',
        'Best candidate (sample',
        'SELECTED: Sample'
    ]
    is_complete = all(marker in content for marker in markers)
    
    sample_scores = extract_sample_scores(content)
    
    return is_complete, sample_scores

def analyze_all_tasks(logs_dir):
    """Analyze completion status for all tasks by grouping files by task ID."""
    
    # Dictionary to store results by task_id
    tasks = defaultdict(lambda: {
        'task_id': None,
        'hypothesis_formation': {},
        'hypothesis_validation': {},
        'final_code': None,
        'final_code_scores': {},
        'repair_code': None,
        'repair_code_scores': {}
    })
    
    # Process all .txt files
    txt_files = [f for f in logs_dir.iterdir() if f.is_file() and f.suffix == '.txt']
    
    for file_path in txt_files:
        try:
            content = file_path.read_text(encoding='utf-8')
            filename = file_path.name
            
            # Extract task_id from filename (first part before underscore)
            # Format: {task_id}_sample{N}_phase2a_hypothesis.txt
            #         {task_id}_sample{N}_phase2b_validation.txt
            #         {task_id}_selection_summary.txt
            #         {task_id}_repair_selection_summary.txt
            
            task_id_match = re.match(r'^([a-f0-9]+)_', filename)
            if not task_id_match:
                continue
            
            task_id = task_id_match.group(1)
            tasks[task_id]['task_id'] = task_id
            
            # Determine file type and process
            if '_sample' in filename and '_phase2a_' in filename:
                # Hypothesis formation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    tasks[task_id]['hypothesis_formation'][sample_num] = check_hypothesis_formation(content)
            
            elif '_sample' in filename and '_phase2b_' in filename:
                # Hypothesis validation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    tasks[task_id]['hypothesis_validation'][sample_num] = check_hypothesis_validation(content)
            
            elif '_selection_summary.txt' in filename:
                if '_repair_selection_summary.txt' in filename:
                    # Repair selection summary
                    is_complete, scores = check_final_code(content)
                    tasks[task_id]['repair_code'] = is_complete
                    tasks[task_id]['repair_code_scores'] = scores
                else:
                    # Regular selection summary
                    is_complete, scores = check_final_code(content)
                    tasks[task_id]['final_code'] = is_complete
                    tasks[task_id]['final_code_scores'] = scores
        
        except Exception as e:
            print(f"  Error reading {file_path.name}: {e}")
    
    return tasks

def format_sample_dict(sample_dict, expected_samples=4):
    """Format sample completion status."""
    if not sample_dict:
        return "❌ Not found"
    
    status_parts = []
    all_complete = True
    
    for i in range(expected_samples):
        if i in sample_dict:
            if sample_dict[i]:
                status_parts.append(f"S{i}:✓")
            else:
                status_parts.append(f"S{i}:✗")
                all_complete = False
        else:
            status_parts.append(f"S{i}:❌")
            all_complete = False
    
    status_str = " ".join(status_parts)
    summary = "✓ All complete" if all_complete else "✗ Incomplete"
    return f"{status_str} ({summary})"

def format_scores(scores, expected_samples=4):
    """Format sample scores."""
    if not scores:
        return "No scores found"
    
    score_parts = []
    for i in range(expected_samples):
        if i in scores:
            score_parts.append(f"S{i}:{scores[i]:.2f}")
        else:
            score_parts.append(f"S{i}:N/A")
    
    return " ".join(score_parts)

def format_status(status):
    """Format the status for display."""
    if status is None:
        return "❌ Not found"
    elif status is True:
        return "✓ Complete"
    else:
        return "✗ Incomplete"

def get_project_root():
    """Find the project root directory."""
    # Try to get the directory of the current script
    try:
        # If __file__ is available
        current_file = Path(__file__).resolve()
        current_dir = current_file.parent
    except NameError:
        # If __file__ is not available, use current working directory
        current_dir = Path.cwd()
    
    # If we're in src/, go up one level
    if current_dir.name == 'src':
        return current_dir.parent
    
    # Otherwise, assume we're already at project root
    return current_dir

def main():
    # Navigate to logs directory
    project_root = get_project_root()
    logs_dir = project_root / 'logs' / 'grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728'
    
    if not logs_dir.exists():
        print(f"Error: Directory not found: {logs_dir}")
        return
    
    print(f"Analyzing tasks in: {logs_dir}\n")
    print("=" * 100)
    
    # Analyze all tasks
    tasks = analyze_all_tasks(logs_dir)
    
    if not tasks:
        print("No tasks found!")
        return
    
    # Sort task IDs for consistent output
    task_ids = sorted(tasks.keys())
    
    incomplete_tasks = []
    
    for task_id in task_ids:
        results = tasks[task_id]
        
        # Check if all components are complete
        hyp_form_complete = len(results['hypothesis_formation']) == 4 and all(results['hypothesis_formation'].values())
        hyp_val_complete = len(results['hypothesis_validation']) == 4 and all(results['hypothesis_validation'].values())
        final_code_complete = results['final_code'] is True
        
        all_complete = hyp_form_complete and hyp_val_complete and final_code_complete
        
        if not all_complete:
            incomplete_tasks.append(results)
        
        # Print status
        status_symbol = "✓" if all_complete else "✗"
        print(f"{status_symbol} Task {task_id}:")
        print(f"  Hypothesis Formation (Phase 2A): {format_sample_dict(results['hypothesis_formation'])}")
        print(f"  Hypothesis Validation (Phase 2B): {format_sample_dict(results['hypothesis_validation'])}")
        print(f"  Final Code: {format_status(results['final_code'])}")
        if results['final_code_scores']:
            print(f"    Sample Scores: {format_scores(results['final_code_scores'])}")
        
        # Print repair code info if exists
        if results['repair_code'] is not None:
            print(f"  Repair Code: {format_status(results['repair_code'])}")
            if results['repair_code_scores']:
                print(f"    Sample Scores: {format_scores(results['repair_code_scores'])}")
        
        print()
    
    # Summary
    print("=" * 100)
    print(f"\nSummary:")
    print(f"  Total tasks: {len(tasks)}")
    print(f"  Complete: {len(tasks) - len(incomplete_tasks)}")
    print(f"  Incomplete: {len(incomplete_tasks)}")
    
    if incomplete_tasks:
        print("\nIncomplete tasks details:")
        for result in incomplete_tasks:
            missing = []
            
            # Check what's missing for hypothesis formation
            hyp_form = result['hypothesis_formation']
            if len(hyp_form) < 4:
                missing_samples = [i for i in range(4) if i not in hyp_form]
                missing.append(f"2a(missing S{missing_samples})")
            elif not all(hyp_form.values()):
                incomplete_samples = [i for i in range(4) if i in hyp_form and not hyp_form[i]]
                missing.append(f"2a(incomplete S{incomplete_samples})")
            
            # Check what's missing for hypothesis validation
            hyp_val = result['hypothesis_validation']
            if len(hyp_val) < 4:
                missing_samples = [i for i in range(4) if i not in hyp_val]
                missing.append(f"2b(missing S{missing_samples})")
            elif not all(hyp_val.values()):
                incomplete_samples = [i for i in range(4) if i in hyp_val and not hyp_val[i]]
                missing.append(f"2b(incomplete S{incomplete_samples})")
            
            # Check final code
            if not result['final_code']:
                missing.append('code')
            
            print(f"  {result['task_id']}: {', '.join(missing) if missing else 'unknown issue'}")

if __name__ == '__main__':
    main()

Analyzing tasks in: /home/te0245/llms_ftw/logs/grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728

✗ Task 0934a4d8:
  Hypothesis Formation (Phase 2A): S0:✗ S1:✗ S2:✓ S3:✓ (✗ Incomplete)
  Hypothesis Validation (Phase 2B): S0:✓ S1:✗ S2:✗ S3:✓ (✗ Incomplete)
  Final Code: ✓ Complete
    Sample Scores: S0:0.79 S1:0.00 S2:0.79 S3:0.35
  Repair Code: ✓ Complete
    Sample Scores: S0:0.00 S1:0.00 S2:0.00 S3:0.35

✗ Task 135a2760:
  Hypothesis Formation (Phase 2A): S0:✓ S1:✓ S2:✗ S3:✓ (✗ Incomplete)
  Hypothesis Validation (Phase 2B): S0:✓ S1:✓ S2:✓ S3:✗ (✗ Incomplete)
  Final Code: ✓ Complete
    Sample Scores: S0:1.00 S1:0.00 S2:0.75 S3:1.00

✗ Task 136b0064:
  Hypothesis Formation (Phase 2A): S0:✓ S1:✓ S2:✓ S3:✓ (✓ All complete)
  Hypothesis Validation (Phase 2B): S0:✓ S1:✓ S2:✗ S3:✓ (✗ Incomplete)
  Final Code: ✓ Complete
    Sample Scores: S0:1.00 S1:0.83 S2:0.00 S3:0.33

✗ Task 13e47133:
  Hypothesis Formation (Phase 2A): S0:✓ S1:✓ S2:✗ S3:✗ (✗ Incomplete)
  Hypothesis Validation (Pha

In [20]:
import os
from pathlib import Path
import re
from collections import defaultdict

def check_hypothesis_formation(content):
    """Check if hypothesis formation file is complete."""
    return content.strip().endswith('</pattern_summary>')

def check_hypothesis_validation(content):
    """Check if hypothesis validation file is complete."""
    return '</validated_pattern>' in content

def extract_sample_scores(content):
    """Extract sample scores from final code file."""
    scores = {}
    
    # Look for the sample scores section
    # Pattern: "Sample 0: 1.00 ✓" or "Sample 0: 1.00" or "Sample 0: 0.67"
    pattern = r'Sample (\d+):\s+([\d.]+)'
    matches = re.findall(pattern, content)
    
    for sample_num, score in matches:
        scores[int(sample_num)] = float(score)
    
    return scores

def check_final_code(content):
    """Check if final code file is complete and extract sample scores."""
    # Look for the test performance section that indicates completion
    markers = [
        'Test Set Performance:',
        'Best candidate (sample',
        'SELECTED: Sample'
    ]
    is_complete = all(marker in content for marker in markers)
    
    sample_scores = extract_sample_scores(content)
    
    return is_complete, sample_scores

def analyze_all_tasks(logs_dir):
    """Analyze completion status for all tasks by grouping files by task ID."""
    
    # Dictionary to store results by task_id
    tasks = defaultdict(lambda: {
        'task_id': None,
        'hypothesis_formation': {},
        'hypothesis_validation': {},
        'final_code': None,
        'final_code_scores': {},
        'repair_code': None,
        'repair_code_scores': {}
    })
    
    # Process all .txt files
    txt_files = [f for f in logs_dir.iterdir() if f.is_file() and f.suffix == '.txt']
    
    for file_path in txt_files:
        try:
            content = file_path.read_text(encoding='utf-8')
            filename = file_path.name
            
            # Extract task_id from filename (first part before underscore)
            task_id_match = re.match(r'^([a-f0-9]+)_', filename)
            if not task_id_match:
                continue
            
            task_id = task_id_match.group(1)
            tasks[task_id]['task_id'] = task_id
            
            # Determine file type and process
            if '_sample' in filename and '_phase2a_' in filename:
                # Hypothesis formation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    tasks[task_id]['hypothesis_formation'][sample_num] = check_hypothesis_formation(content)
            
            elif '_sample' in filename and '_phase2b_' in filename:
                # Hypothesis validation file
                sample_match = re.search(r'_sample(\d+)_', filename)
                if sample_match:
                    sample_num = int(sample_match.group(1))
                    tasks[task_id]['hypothesis_validation'][sample_num] = check_hypothesis_validation(content)
            
            elif '_selection_summary.txt' in filename:
                if '_repair_selection_summary.txt' in filename:
                    # Repair selection summary
                    is_complete, scores = check_final_code(content)
                    tasks[task_id]['repair_code'] = is_complete
                    tasks[task_id]['repair_code_scores'] = scores
                else:
                    # Regular selection summary
                    is_complete, scores = check_final_code(content)
                    tasks[task_id]['final_code'] = is_complete
                    tasks[task_id]['final_code_scores'] = scores
        
        except Exception as e:
            print(f"  Error reading {file_path.name}: {e}")
    
    return tasks

def format_sample_status_compact(sample_dict, expected_samples=4):
    """Format sample completion status compactly for table."""
    if not sample_dict:
        return "❌"
    
    status_parts = []
    for i in range(expected_samples):
        if i in sample_dict:
            status_parts.append("✓" if sample_dict[i] else "✗")
        else:
            status_parts.append("❌")
    
    return " ".join(status_parts)

def format_scores_compact(scores, expected_samples=4):
    """Format sample scores compactly for table."""
    if not scores:
        return "-"
    
    score_parts = []
    for i in range(expected_samples):
        if i in scores:
            score_parts.append(f"{scores[i]:.2f}")
        else:
            score_parts.append("-")
    
    return " | ".join(score_parts)

def get_overall_status(results):
    """Get overall completion status."""
    hyp_form_complete = len(results['hypothesis_formation']) == 4 and all(results['hypothesis_formation'].values())
    hyp_val_complete = len(results['hypothesis_validation']) == 4 and all(results['hypothesis_validation'].values())
    final_code_complete = results['final_code'] is True
    
    if hyp_form_complete and hyp_val_complete and final_code_complete:
        return "✓"
    else:
        return "✗"

def get_project_root():
    """Find the project root directory."""
    try:
        current_file = Path(__file__).resolve()
        current_dir = current_file.parent
    except NameError:
        current_dir = Path.cwd()
    
    if current_dir.name == 'src':
        return current_dir.parent
    
    return current_dir

def format_sample_status_individual(sample_dict, sample_num):
    """Format individual sample completion status."""
    if sample_num not in sample_dict:
        return "❌"
    return "✓" if sample_dict[sample_num] else "✗"

def format_score_individual(scores, sample_num):
    """Format individual sample score."""
    if sample_num not in scores:
        return "-"
    return f"{scores[sample_num]:.2f}"

def analyze_pipeline_efficiency(tasks):
    """Analyze where in the pipeline tasks are failing and resource waste."""
    
    stats = {
        'total_tasks': len(tasks),
        'total_samples': len(tasks) * 4,  # 4 samples per task
        
        # Phase 2A failures
        '2a_missing': 0,
        '2a_incomplete': 0,
        '2a_complete': 0,
        
        # Phase 2B given 2A status
        '2b_after_2a_missing': 0,  # 2B attempted even though 2A missing
        '2b_after_2a_incomplete': 0,  # 2B attempted even though 2A incomplete
        '2b_after_2a_complete_missing': 0,  # 2B missing despite 2A complete
        '2b_after_2a_complete_incomplete': 0,  # 2B incomplete despite 2A complete
        '2b_after_2a_complete_complete': 0,  # 2B complete and 2A complete
        
        # Code given 2A/2B status
        'code_missing': 0,
        'code_complete': 0,
        'code_complete_despite_2a_issues': 0,  # Code complete but 2A has issues
        'code_complete_despite_2b_issues': 0,  # Code complete but 2B has issues
        
        # Sample-level breakdown
        'sample_breakdown': {
            '2a_only_fail': 0,  # 2A fails, 2B succeeds
            '2b_only_fail': 0,  # 2A succeeds, 2B fails
            'both_fail': 0,  # Both 2A and 2B fail
            'both_succeed': 0,  # Both 2A and 2B succeed
            '2a_missing_2b_present': 0,  # 2A file missing but 2B file exists
        }
    }
    
    # Analyze each task
    for task_id, results in tasks.items():
        # Check if code exists
        if results['final_code']:
            stats['code_complete'] += 1
        else:
            stats['code_missing'] += 1
        
        # Analyze each of the 4 samples
        for sample_num in range(4):
            # Phase 2A status
            has_2a = sample_num in results['hypothesis_formation']
            is_2a_complete = results['hypothesis_formation'].get(sample_num, False)
            
            # Phase 2B status
            has_2b = sample_num in results['hypothesis_validation']
            is_2b_complete = results['hypothesis_validation'].get(sample_num, False)
            
            # Count 2A outcomes
            if not has_2a:
                stats['2a_missing'] += 1
            elif not is_2a_complete:
                stats['2a_incomplete'] += 1
            else:
                stats['2a_complete'] += 1
            
            # Count 2B outcomes based on 2A status
            if not has_2a:
                if has_2b:
                    stats['2b_after_2a_missing'] += 1
                    stats['sample_breakdown']['2a_missing_2b_present'] += 1
            elif not is_2a_complete:
                if has_2b:
                    stats['2b_after_2a_incomplete'] += 1
            else:  # 2A is complete
                if not has_2b:
                    stats['2b_after_2a_complete_missing'] += 1
                elif not is_2b_complete:
                    stats['2b_after_2a_complete_incomplete'] += 1
                else:
                    stats['2b_after_2a_complete_complete'] += 1
            
            # Sample-level breakdown
            if is_2a_complete and is_2b_complete:
                stats['sample_breakdown']['both_succeed'] += 1
            elif is_2a_complete and not is_2b_complete:
                stats['sample_breakdown']['2b_only_fail'] += 1
            elif not is_2a_complete and is_2b_complete:
                stats['sample_breakdown']['2a_only_fail'] += 1
            elif not is_2a_complete and not is_2b_complete:
                stats['sample_breakdown']['both_fail'] += 1
        
        # Check if code completed despite 2A/2B issues
        if results['final_code']:
            has_2a_issues = len(results['hypothesis_formation']) < 4 or not all(results['hypothesis_formation'].values())
            has_2b_issues = len(results['hypothesis_validation']) < 4 or not all(results['hypothesis_validation'].values())
            
            if has_2a_issues:
                stats['code_complete_despite_2a_issues'] += 1
            if has_2b_issues:
                stats['code_complete_despite_2b_issues'] += 1
    
    return stats

def print_efficiency_report(stats):
    """Print a detailed efficiency report."""
    print("\n" + "=" * 100)
    print("PIPELINE EFFICIENCY ANALYSIS")
    print("=" * 100)
    
    print(f"\n📊 OVERALL STATISTICS")
    print(f"   Total Tasks: {stats['total_tasks']}")
    print(f"   Total Samples (4 per task): {stats['total_samples']}")
    
    print(f"\n🔴 PHASE 2A (Hypothesis Formation) - Sample Level")
    print(f"   Complete: {stats['2a_complete']:4d} / {stats['total_samples']} ({stats['2a_complete']/stats['total_samples']*100:.1f}%)")
    print(f"   Incomplete: {stats['2a_incomplete']:4d} / {stats['total_samples']} ({stats['2a_incomplete']/stats['total_samples']*100:.1f}%) ⚠️  WASTED RESOURCES")
    print(f"   Missing: {stats['2a_missing']:4d} / {stats['total_samples']} ({stats['2a_missing']/stats['total_samples']*100:.1f}%)")
    
    print(f"\n🟡 PHASE 2B (Hypothesis Validation) - Given 2A Status")
    print(f"   2B attempted when 2A missing: {stats['2b_after_2a_missing']:4d} ⚠️  WASTED RESOURCES")
    print(f"   2B attempted when 2A incomplete: {stats['2b_after_2a_incomplete']:4d} ⚠️  WASTED RESOURCES")
    print(f"   2B missing despite 2A complete: {stats['2b_after_2a_complete_missing']:4d} 🔴 PIPELINE BREAK")
    print(f"   2B incomplete despite 2A complete: {stats['2b_after_2a_complete_incomplete']:4d} ⚠️  WASTED RESOURCES")
    print(f"   2B complete when 2A complete: {stats['2b_after_2a_complete_complete']:4d} ✅ SUCCESS")
    
    print(f"\n🟢 CODE GENERATION")
    print(f"   Code complete: {stats['code_complete']:4d} / {stats['total_tasks']} ({stats['code_complete']/stats['total_tasks']*100:.1f}%)")
    print(f"   Code missing: {stats['code_missing']:4d} / {stats['total_tasks']} ({stats['code_missing']/stats['total_tasks']*100:.1f}%)")
    print(f"   Code complete despite 2A issues: {stats['code_complete_despite_2a_issues']:4d}")
    print(f"   Code complete despite 2B issues: {stats['code_complete_despite_2b_issues']:4d}")
    
    print(f"\n📈 SAMPLE-LEVEL BREAKDOWN ({stats['total_samples']} samples)")
    bd = stats['sample_breakdown']
    print(f"   Both 2A & 2B succeed: {bd['both_succeed']:4d} ({bd['both_succeed']/stats['total_samples']*100:.1f}%) ✅")
    print(f"   Only 2A fails: {bd['2a_only_fail']:4d} ({bd['2a_only_fail']/stats['total_samples']*100:.1f}%) 🔴")
    print(f"   Only 2B fails: {bd['2b_only_fail']:4d} ({bd['2b_only_fail']/stats['total_samples']*100:.1f}%) 🟡")
    print(f"   Both fail: {bd['both_fail']:4d} ({bd['both_fail']/stats['total_samples']*100:.1f}%) 🔴🔴")
    print(f"   2A missing but 2B present: {bd['2a_missing_2b_present']:4d} (shouldn't happen!)")
    
    # Calculate waste
    total_waste = (stats['2a_incomplete'] + 
                   stats['2b_after_2a_missing'] + 
                   stats['2b_after_2a_incomplete'] + 
                   stats['2b_after_2a_complete_incomplete'])
    
    print(f"\n💰 RESOURCE WASTE ESTIMATION")
    print(f"   Total wasted sample runs: {total_waste} / {stats['total_samples']} ({total_waste/stats['total_samples']*100:.1f}%)")
    print(f"   Breakdown:")
    print(f"      - 2A incomplete (ran but failed): {stats['2a_incomplete']}")
    print(f"      - 2B ran when 2A missing: {stats['2b_after_2a_missing']}")
    print(f"      - 2B ran when 2A incomplete: {stats['2b_after_2a_incomplete']}")
    print(f"      - 2B incomplete when 2A complete: {stats['2b_after_2a_complete_incomplete']}")
    
    print("\n" + "=" * 100)

def analyze_detailed_statistics(tasks):
    """Analyze detailed statistics about repair improvements and validation coverage."""
    
    stats = {
        # Repair effectiveness
        'repair_improved': 0,  # Repair made at least one sample better
        'repair_degraded': 0,  # Repair made at least one sample worse
        'repair_no_change': 0,  # Repair existed but no improvement
        'already_perfect': 0,  # All train scores = 1.00, no repair needed
        'repair_made_perfect': 0,  # Repair achieved at least one 1.00 where train didn't
        
        # Validation coverage
        'code_with_no_validations': 0,  # Code exists but no 2B validations complete
        'code_with_partial_validations': 0,  # Code exists with 1-3 2B complete
        'code_with_all_validations': 0,  # Code exists with all 4 2B complete
        
        # Hypothesis coverage
        'all_4_hypothesis_complete': 0,  # All 4 2A samples complete
        'all_4_validation_complete': 0,  # All 4 2B samples complete
        'all_4_both_complete': 0,  # All 4 2A and 2B complete
        
        # Sample-level repair analysis
        'samples_improved_by_repair': 0,  # Individual samples where repair > train
        'samples_degraded_by_repair': 0,  # Individual samples where repair < train
        'samples_unchanged_by_repair': 0,  # Individual samples where repair == train
    }
    
    for task_id, results in tasks.items():
        # Check hypothesis coverage
        hyp_form = results['hypothesis_formation']
        hyp_val = results['hypothesis_validation']
        
        if len(hyp_form) == 4 and all(hyp_form.values()):
            stats['all_4_hypothesis_complete'] += 1
        
        if len(hyp_val) == 4 and all(hyp_val.values()):
            stats['all_4_validation_complete'] += 1
        
        if (len(hyp_form) == 4 and all(hyp_form.values()) and 
            len(hyp_val) == 4 and all(hyp_val.values())):
            stats['all_4_both_complete'] += 1
        
        # Check code with validation coverage
        if results['final_code']:
            complete_validations = sum(1 for i in range(4) if hyp_val.get(i, False))
            
            if complete_validations == 0:
                stats['code_with_no_validations'] += 1
            elif complete_validations < 4:
                stats['code_with_partial_validations'] += 1
            else:
                stats['code_with_all_validations'] += 1
        
        # Analyze repair effectiveness
        if results['repair_code'] and results['repair_code_scores'] and results['final_code_scores']:
            train_scores = results['final_code_scores']
            repair_scores = results['repair_code_scores']
            
            # Check if already perfect
            if all(train_scores.get(i, 0) == 1.00 for i in range(4) if i in train_scores):
                stats['already_perfect'] += 1
            
            # Compare repair vs train
            task_improved = False
            task_degraded = False
            task_made_perfect = False
            
            for i in range(4):
                if i in train_scores and i in repair_scores:
                    train = train_scores[i]
                    repair = repair_scores[i]
                    
                    if repair > train:
                        stats['samples_improved_by_repair'] += 1
                        task_improved = True
                        if repair == 1.00 and train < 1.00:
                            task_made_perfect = True
                    elif repair < train:
                        stats['samples_degraded_by_repair'] += 1
                        task_degraded = True
                    else:
                        stats['samples_unchanged_by_repair'] += 1
            
            if task_improved:
                stats['repair_improved'] += 1
            if task_degraded:
                stats['repair_degraded'] += 1
            if not task_improved and not task_degraded:
                stats['repair_no_change'] += 1
            if task_made_perfect:
                stats['repair_made_perfect'] += 1
    
    return stats

def print_detailed_statistics(stats):
    """Print detailed statistics report."""
    print("\n" + "=" * 100)
    print("DETAILED STATISTICS ANALYSIS")
    print("=" * 100)
    
    print(f"\n🔧 REPAIR EFFECTIVENESS (Task-Level)")
    tasks_with_repair = stats['repair_improved'] + stats['repair_degraded'] + stats['repair_no_change']
    if tasks_with_repair > 0:
        print(f"   Tasks with repair that improved scores: {stats['repair_improved']:4d} / {tasks_with_repair} ({stats['repair_improved']/tasks_with_repair*100:.1f}%)")
        print(f"   Tasks with repair that degraded scores: {stats['repair_degraded']:4d} / {tasks_with_repair} ({stats['repair_degraded']/tasks_with_repair*100:.1f}%)")
        print(f"   Tasks with repair but no change: {stats['repair_no_change']:4d} / {tasks_with_repair} ({stats['repair_no_change']/tasks_with_repair*100:.1f}%)")
        print(f"   Tasks already perfect (all train=1.00): {stats['already_perfect']:4d}")
        print(f"   Tasks where repair achieved perfect (1.00): {stats['repair_made_perfect']:4d}")
    else:
        print(f"   No tasks with repair found")
    
    print(f"\n📊 REPAIR EFFECTIVENESS (Sample-Level)")
    total_repair_samples = (stats['samples_improved_by_repair'] + 
                            stats['samples_degraded_by_repair'] + 
                            stats['samples_unchanged_by_repair'])
    if total_repair_samples > 0:
        print(f"   Samples improved by repair: {stats['samples_improved_by_repair']:4d} / {total_repair_samples} ({stats['samples_improved_by_repair']/total_repair_samples*100:.1f}%)")
        print(f"   Samples degraded by repair: {stats['samples_degraded_by_repair']:4d} / {total_repair_samples} ({stats['samples_degraded_by_repair']/total_repair_samples*100:.1f}%)")
        print(f"   Samples unchanged by repair: {stats['samples_unchanged_by_repair']:4d} / {total_repair_samples} ({stats['samples_unchanged_by_repair']/total_repair_samples*100:.1f}%)")
    
    print(f"\n✅ VALIDATION COVERAGE (for tasks with code)")
    total_with_code = (stats['code_with_no_validations'] + 
                       stats['code_with_partial_validations'] + 
                       stats['code_with_all_validations'])
    if total_with_code > 0:
        print(f"   Code with NO complete validations (2B): {stats['code_with_no_validations']:4d} / {total_with_code} ({stats['code_with_no_validations']/total_with_code*100:.1f}%) ⚠️")
        print(f"   Code with PARTIAL validations (1-3): {stats['code_with_partial_validations']:4d} / {total_with_code} ({stats['code_with_partial_validations']/total_with_code*100:.1f}%)")
        print(f"   Code with ALL validations (4): {stats['code_with_all_validations']:4d} / {total_with_code} ({stats['code_with_all_validations']/total_with_code*100:.1f}%) ✅")
    
    print(f"\n🎯 COMPLETE COVERAGE")
    total_tasks = 120  # From your data
    print(f"   Tasks with all 4 hypothesis (2A) complete: {stats['all_4_hypothesis_complete']:4d} / {total_tasks} ({stats['all_4_hypothesis_complete']/total_tasks*100:.1f}%)")
    print(f"   Tasks with all 4 validation (2B) complete: {stats['all_4_validation_complete']:4d} / {total_tasks} ({stats['all_4_validation_complete']/total_tasks*100:.1f}%)")
    print(f"   Tasks with all 4 BOTH (2A+2B) complete: {stats['all_4_both_complete']:4d} / {total_tasks} ({stats['all_4_both_complete']/total_tasks*100:.1f}%) ✅")
    
    print("\n" + "=" * 100)
    
        
def main():
    # Navigate to logs directory
    project_root = get_project_root()
    logs_dir = project_root / 'logs' / 'grok4.1fast_similar_notrainrepair_2a2b_prog_k4_82728'
    
    if not logs_dir.exists():
        print(f"Error: Directory not found: {logs_dir}")
        return
    
    # Analyze all tasks
    tasks = analyze_all_tasks(logs_dir)
    
    if not tasks:
        print("No tasks found!")
        return
    
    # Sort task IDs for consistent output
    task_ids = sorted(tasks.keys())
    
    # Analyze pipeline efficiency FIRST
    stats = analyze_pipeline_efficiency(tasks)
    
    # Print efficiency report BEFORE creating the markdown file
    print_efficiency_report(stats)
    
    # Add detailed statistics
    detailed_stats = analyze_detailed_statistics(tasks)
    print_detailed_statistics(detailed_stats)
    
    # Create markdown table
    output_file = project_root / 'task_completion_report.md'
    
    complete_count = 0
    incomplete_count = 0
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("# Task Completion Report\n\n")
        f.write(f"Total Tasks: {len(tasks)}\n\n")
        
        # Write table header
        f.write("| Status | Task ID | 2A-S0 | 2A-S1 | 2A-S2 | 2A-S3 | 2B-S0 | 2B-S1 | 2B-S2 | 2B-S3 | Code | Train-S0 | Train-S1 | Train-S2 | Train-S3 | Repair | Rep-S0 | Rep-S1 | Rep-S2 | Rep-S3 |\n")
        f.write("|--------|---------|-------|-------|-------|-------|-------|-------|-------|-------|------|----------|----------|----------|----------|--------|--------|--------|--------|--------|\n")
        
        for task_id in task_ids:
            results = tasks[task_id]
            overall_status = get_overall_status(results)
            
            if overall_status == "✓":
                complete_count += 1
            else:
                incomplete_count += 1
            
            # Format individual cells
            row = [
                overall_status,
                f"`{task_id}`",
            ]
            
            # Phase 2A samples
            for i in range(4):
                row.append(format_sample_status_individual(results['hypothesis_formation'], i))
            
            # Phase 2B samples
            for i in range(4):
                row.append(format_sample_status_individual(results['hypothesis_validation'], i))
            
            # Code status
            row.append("✓" if results['final_code'] else "❌")
            
            # Training scores
            for i in range(4):
                row.append(format_score_individual(results['final_code_scores'], i))
            
            # Repair status
            row.append("✓" if results['repair_code'] else ("-" if results['repair_code'] is None else "✗"))
            
            # Repair scores
            for i in range(4):
                if results['repair_code_scores']:
                    row.append(format_score_individual(results['repair_code_scores'], i))
                else:
                    row.append("-")
            
            f.write("| " + " | ".join(row) + " |\n")
        
        # Write summary
        f.write(f"\n## Summary\n\n")
        f.write(f"- **Total Tasks**: {len(tasks)}\n")
        f.write(f"- **Complete**: {complete_count}\n")
        f.write(f"- **Incomplete**: {incomplete_count}\n")
        
        # Write legend
        f.write(f"\n## Legend\n\n")
        f.write(f"- **Status**: ✓ = All complete, ✗ = Has issues\n")
        f.write(f"- **2A/2B Columns**: ✓ = Complete, ✗ = Incomplete, ❌ = Missing file\n")
        f.write(f"- **Code/Repair**: ✓ = Complete, ✗ = Incomplete, ❌ = Missing, - = Not applicable\n")
        f.write(f"- **Train/Rep Scores**: 0.00-1.00 = Training accuracy, - = Not available\n")
    
    print(f"\n✓ Markdown table written to: {output_file}")
    
    print(f"\nBasic Summary:")
    print(f"  Total tasks: {len(tasks)}")
    print(f"  Complete: {complete_count}")
    print(f"  Incomplete: {incomplete_count}")
    
    # Print ALL tasks to console in cleaner format with repair scores
    print(f"\nAll {len(task_ids)} tasks:")
    print("=" * 180)
    print(f"{'St':<2} {'Task ID':<10} {'2A (S0 S1 S2 S3)':<18} {'2B (S0 S1 S2 S3)':<18} {'Code':<4} {'Train Scores (S0   S1   S2   S3)':<35} {'Rep':<3} {'Repair Scores (S0   S1   S2   S3)':<35}")
    print("-" * 180)
    
    for task_id in task_ids:
        results = tasks[task_id]
        overall_status = get_overall_status(results)
        
        # Format phase 2A
        phase2a = " ".join([format_sample_status_individual(results['hypothesis_formation'], i) for i in range(4)])
        
        # Format phase 2B
        phase2b = " ".join([format_sample_status_individual(results['hypothesis_validation'], i) for i in range(4)])
        
        # Format code status
        code_status = "✓" if results['final_code'] else "❌"
        
        # Format train scores
        train_scores = "  ".join([format_score_individual(results['final_code_scores'], i) for i in range(4)])
        
        # Format repair status
        repair_status = "✓" if results['repair_code'] else ("-" if results['repair_code'] is None else "✗")
        
        # Format repair scores
        if results['repair_code_scores']:
            repair_scores = "  ".join([format_score_individual(results['repair_code_scores'], i) for i in range(4)])
        else:
            repair_scores = "-     -     -     -"
        
        print(f"{overall_status:<2} {task_id:<10} {phase2a:<18} {phase2b:<18} {code_status:<4} {train_scores:<35} {repair_status:<3} {repair_scores:<35}")

if __name__ == '__main__':
    main()


PIPELINE EFFICIENCY ANALYSIS

📊 OVERALL STATISTICS
   Total Tasks: 120
   Total Samples (4 per task): 480

🔴 PHASE 2A (Hypothesis Formation) - Sample Level
   Complete:  326 / 480 (67.9%)
   Incomplete:  154 / 480 (32.1%) ⚠️  WASTED RESOURCES
   Missing:    0 / 480 (0.0%)

🟡 PHASE 2B (Hypothesis Validation) - Given 2A Status
   2B attempted when 2A missing:    0 ⚠️  WASTED RESOURCES
   2B attempted when 2A incomplete:  154 ⚠️  WASTED RESOURCES
   2B missing despite 2A complete:    0 🔴 PIPELINE BREAK
   2B incomplete despite 2A complete:   64 ⚠️  WASTED RESOURCES
   2B complete when 2A complete:  262 ✅ SUCCESS

🟢 CODE GENERATION
   Code complete:  100 / 120 (83.3%)
   Code missing:   20 / 120 (16.7%)
   Code complete despite 2A issues:   68
   Code complete despite 2B issues:   69

📈 SAMPLE-LEVEL BREAKDOWN (480 samples)
   Both 2A & 2B succeed:  262 (54.6%) ✅
   Only 2A fails:   82 (17.1%) 🔴
   Only 2B fails:   64 (13.3%) 🟡
   Both fail:   72 (15.0%) 🔴🔴
   2A missing but 2B present:   